# N17 CheXchoNet V2
## based on Machine Learning and Deep Learning

This revision separates branch selection, fusion fitting, sigmoid calibration and threshold tuning by patient.
It saves the split manifest, per-examination predictions, fitted models, runtime versions and artifact hashes to your private Drive.
The old test set has already been inspected: it remains labelled **historical comparison**, never a new untouched test.
No V2 image/fusion score is supplied in this notebook before it actually runs.

**Run:** select GPU, run cells in order. Your existing CheXchoNet dataset stays in the same Drive folder.
Start with age and sex metadata; the full five-feature retrospective variant is opt-in because device/transplant flags have unresolved T0 provenance.

In [ ]:
import importlib.util, subprocess, sys
required = {'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn',
            'joblib':'joblib','PIL':'pillow','torch':'torch','torchvision':'torchvision'}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
from pathlib import Path
from datetime import datetime, timezone
import uuid, json
import numpy as np
import pandas as pd
import joblib
import torch
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/Hoc_may_2_0/DL_Xray/CheXchoNet')
    CACHE = Path('/content/chexchonet_v2_cache')
    MODULE_ROOT = Path('/content/n17_chexchonet_v2_modules')
except ImportError:
    PROJECT = Path.cwd()
    CACHE = PROJECT/'chexchonet_v2_cache'
    MODULE_ROOT = PROJECT/'_v2_modules'
MODULE_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — choose GPU for full training')


In [ ]:
MODULE_SOURCES = {'fusion_protocol.py': '"""Patient-separated development and auditable late fusion; no GPU dependency."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nimport sys\nfrom importlib import metadata as packages\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.special import expit, logit\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,\n                             f1_score, fbeta_score, log_loss, precision_score,\n                             recall_score, roc_auc_score)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.utils.class_weight import compute_sample_weight\n\nTARGET = \'composite_slvh_dlv\'\nFEATURES = [\'age_scaled\', \'sex_binary\', \'heart_transplant\', \'lung_transplant\', \'pacemaker_or_icd\']\nKEYS = [\'patient_id\', \'cxr_filename\']\nROLES = (\'train\', \'branch_val\', \'fusion_fit\', \'calibration\', \'threshold\', \'historical_test\')\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef environment_receipt():\n    versions = {}\n    for name in [\'numpy\', \'pandas\', \'scipy\', \'scikit-learn\', \'joblib\', \'torch\', \'torchvision\', \'pillow\']:\n        try:\n            versions[name] = packages.version(name)\n        except packages.PackageNotFoundError:\n            versions[name] = None\n    return {\'python\': sys.version, \'platform\': platform.platform(), \'packages\': versions}\n\n\ndef prepare_frame(raw):\n    required = {*KEYS, \'age\', \'sex\', \'slvh\', \'dlv\', TARGET}\n    if not required.issubset(raw.columns):\n        raise ValueError(f\'Missing required fields: {sorted(required-set(raw.columns))}\')\n    frame = raw.copy()\n    if frame[KEYS].isna().any().any():\n        raise ValueError(\'Missing patient or image key\')\n    for key in KEYS:\n        frame[key] = frame[key].astype(str).str.strip()\n    if frame[KEYS].eq(\'\').any().any() or frame.duplicated(KEYS).any():\n        raise ValueError(\'Blank or duplicate examination keys\')\n    # Keep paths local to the expected images directory.\n    if frame.cxr_filename.str.contains(r\'[/\\\\]|^\\.{1,2}$\', regex=True).any():\n        raise ValueError(\'Image filename must be a basename, not a path\')\n    frame[\'age\'] = pd.to_numeric(frame.age, errors=\'coerce\')\n    frame[\'sex\'] = frame.sex.astype(str).str.upper().str.strip()\n    if not frame.age.between(18, 90).all() or not frame.sex.isin([\'F\', \'M\']).all():\n        raise ValueError(\'Invalid age/sex; inspect exclusions explicitly before training\')\n    for label in [\'slvh\', \'dlv\', TARGET]:\n        frame[label] = pd.to_numeric(frame[label], errors=\'coerce\')\n        if not frame[label].isin([0, 1]).all():\n            raise ValueError(f\'Invalid/missing binary label: {label}\')\n        frame[label] = frame[label].astype(int)\n    if not frame[TARGET].eq(frame.slvh | frame.dlv).all():\n        raise ValueError(\'Composite label disagrees with SLVH OR DLV\')\n    frame[\'age_scaled\'] = (frame.age - 18) / 72\n    frame[\'sex_binary\'] = frame.sex.map({\'F\': 0., \'M\': 1.})\n    for feature in FEATURES[2:]:\n        if feature not in frame:\n            frame[feature] = np.nan\n        frame[feature] = pd.to_numeric(frame[feature], errors=\'coerce\')\n        if not frame[feature].dropna().isin([0, 1]).all():\n            raise ValueError(f\'Non-binary metadata: {feature}\')\n    return frame\n\n\ndef validate_partition(frame):\n    if frame.empty or frame.duplicated(KEYS).any():\n        raise ValueError(\'Empty manifest or duplicate keys\')\n    if frame.role.isna().any() or not set(frame.role).issubset(ROLES):\n        raise ValueError(\'Unrecognised development role\')\n    if frame.groupby(\'patient_id\').role.nunique().max() != 1:\n        raise ValueError(\'Patient overlap across development roles\')\n    if set(frame.role) != set(ROLES):\n        raise ValueError(\'At least one required development role is missing\')\n    for role, subset in frame.groupby(\'role\'):\n        if subset[TARGET].nunique() != 2:\n            raise ValueError(f\'Role {role} needs both classes\')\n\n\ndef make_protocol(raw, *, seed=42):\n    """Preserve the old train/test assignment; partition original val 25/35/20/20.\n\n    A supplied split column must be the historical manifest, not a new favourable\n    random split. Without it, reproduce the original notebook\'s exact split.\n    """\n    frame = prepare_frame(raw)\n    patients = frame.groupby(\'patient_id\', sort=True)[TARGET].max().reset_index()\n    if \'split\' not in frame:\n        train, holdout = train_test_split(patients, train_size=.70, random_state=seed,\n                                         stratify=patients[TARGET])\n        val, test = train_test_split(holdout, train_size=.50, random_state=seed+1,\n                                    stratify=holdout[TARGET])\n        mapping = {**dict.fromkeys(train.patient_id, \'train\'), **dict.fromkeys(val.patient_id, \'val\'),\n                   **dict.fromkeys(test.patient_id, \'test\')}\n        frame[\'split\'] = frame.patient_id.map(mapping)\n    if frame.split.isna().any() or set(frame.split) != {\'train\', \'val\', \'test\'}:\n        raise ValueError(\'Historical split must contain train, val and test only\')\n    if frame.groupby(\'patient_id\').split.nunique().max() != 1:\n        raise ValueError(\'Historical patient leakage\')\n    val_patients = patients[patients.patient_id.isin(frame.loc[frame.split.eq(\'val\'), \'patient_id\'])]\n    branch_val, rest = train_test_split(val_patients, train_size=.25, random_state=seed+10,\n                                        stratify=val_patients[TARGET])\n    fusion_fit, rest = train_test_split(rest, train_size=.35/.75, random_state=seed+11,\n                                        stratify=rest[TARGET])\n    calibration, threshold = train_test_split(rest, train_size=.5, random_state=seed+12,\n                                              stratify=rest[TARGET])\n    role_map = {**dict.fromkeys(frame.loc[frame.split.eq(\'train\'), \'patient_id\'], \'train\'),\n                **dict.fromkeys(frame.loc[frame.split.eq(\'test\'), \'patient_id\'], \'historical_test\')}\n    for name, subset in [(\'branch_val\', branch_val), (\'fusion_fit\', fusion_fit),\n                         (\'calibration\', calibration), (\'threshold\', threshold)]:\n        role_map.update(dict.fromkeys(subset.patient_id, name))\n    frame[\'role\'] = frame.patient_id.map(role_map)\n    validate_partition(frame)\n    return frame\n\n\ndef split_summary(frame):\n    return [{\'role\': role, \'images\': len(part), \'patients\': part.patient_id.nunique(),\n             \'positive_images\': int(part[TARGET].sum()), \'prevalence\': float(part[TARGET].mean())}\n            for role, part in frame.groupby(\'role\', sort=True)]\n\n\ndef fit_metadata(frame, *, seed=42, feature_set=\'age_sex\'):\n    validate_partition(frame)\n    # Age/sex default avoids unverified T0 provenance of report-derived flags.\n    features = FEATURES[:2] if feature_set == \'age_sex\' else FEATURES if feature_set == \'full_retrospective\' else None\n    if features is None:\n        raise ValueError(\'Unknown feature_set\')\n    train = frame[frame.role.eq(\'train\')]\n    val = frame[frame.role.eq(\'branch_val\')]\n    candidates = {\n        \'LogisticRegression\': make_pipeline(SimpleImputer(strategy=\'median\', keep_empty_features=True),\n            StandardScaler(), LogisticRegression(class_weight=\'balanced\', max_iter=2000, random_state=seed)),\n        \'HistGradientBoosting\': make_pipeline(SimpleImputer(strategy=\'median\', keep_empty_features=True),\n            HistGradientBoostingClassifier(learning_rate=.05, max_iter=250, l2_regularization=1.,\n                                           early_stopping=False, random_state=seed))}\n    fitted, comparisons = {}, []\n    for name, model in candidates.items():\n        kwargs = {\'histgradientboostingclassifier__sample_weight\': compute_sample_weight(\'balanced\', train[TARGET])} if name == \'HistGradientBoosting\' else {}\n        model.fit(train[features], train[TARGET], **kwargs)\n        probability = model.predict_proba(val[features])[:, 1]\n        comparisons.append({\'model\': name, \'selection_role\': \'branch_val\', \'images\': len(val),\n                            \'average_precision\': float(average_precision_score(val[TARGET], probability)),\n                            \'roc_auc\': float(roc_auc_score(val[TARGET], probability))})\n        fitted[name] = model\n    winner = max(comparisons, key=lambda row: row[\'average_precision\'])[\'model\']\n    return {\'name\': winner, \'model\': fitted[winner], \'features\': features,\n            \'feature_set\': feature_set, \'comparisons\': comparisons}\n\n\ndef pair_predictions(dl, ml):\n    for label, data, probability in [(\'DL\', dl, \'dl_probability\'), (\'ML\', ml, \'ml_probability\')]:\n        if not {*KEYS, \'target\', probability}.issubset(data.columns):\n            raise ValueError(f\'Missing {label} prediction columns\')\n        if data[KEYS].isna().any().any() or data.duplicated(KEYS).any():\n            raise ValueError(f\'Duplicate/missing {label} keys\')\n        if not data.target.isin([0, 1]).all():\n            raise ValueError(f\'Invalid {label} labels\')\n        if not np.isfinite(data[probability]).all() or not data[probability].between(0, 1).all():\n            raise ValueError(f\'Invalid {label} probabilities\')\n    merged = dl.merge(ml[[*KEYS, \'target\', \'ml_probability\']], on=KEYS, how=\'outer\',\n                      validate=\'one_to_one\', indicator=True, suffixes=(\'\', \'_ml\'))\n    if not merged[\'_merge\'].eq(\'both\').all() or not merged.target.eq(merged.target_ml).all():\n        raise ValueError(\'Branch key sets or labels do not match\')\n    return merged.drop(columns=[\'_merge\', \'target_ml\'])\n\n\ndef fit_fusion(frame, *, seed=42):\n    fit = frame[frame.role.eq(\'fusion_fit\')]\n    if fit.target.nunique() != 2:\n        raise ValueError(\'Fusion fitting needs both classes\')\n    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=seed))\n    model.fit(fit[[\'ml_probability\', \'dl_probability\']], fit.target)\n    return model\n\n\ndef fit_calibrators(frame, fusion, *, seed=42):\n    cal = frame[frame.role.eq(\'calibration\')]\n    raw = raw_scores(cal, fusion)\n    if cal.target.nunique() != 2:\n        raise ValueError(\'Calibration needs both classes\')\n    result = {}\n    for name, probability in raw.items():\n        # Unweighted sigmoid calibration on disjoint, natural-prevalence rows.\n        model = LogisticRegression(C=1e6, max_iter=2000, random_state=seed)\n        model.fit(logit(np.clip(probability, 1e-6, 1-1e-6)).reshape(-1, 1), cal.target)\n        result[name] = model\n    return result\n\n\ndef raw_scores(frame, fusion):\n    return {\'ML\': frame.ml_probability.to_numpy(), \'DL\': frame.dl_probability.to_numpy(),\n            \'Fusion\': fusion.predict_proba(frame[[\'ml_probability\', \'dl_probability\']])[:, 1]}\n\n\ndef calibrated_scores(frame, fusion, calibrators):\n    return {name: calibrators[name].predict_proba(logit(np.clip(score, 1e-6, 1-1e-6)).reshape(-1, 1))[:, 1]\n            for name, score in raw_scores(frame, fusion).items()}\n\n\ndef select_threshold(y, probability, *, minimum_specificity=.80):\n    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)\n    if not 0 <= minimum_specificity <= 1 or len(np.unique(y)) != 2:\n        raise ValueError(\'Invalid specificity or threshold labels\')\n    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):\n        raise ValueError(\'Invalid probabilities\')\n    # Exact candidates and cumulative counts avoid O(n^2) threshold scans.\n    order = np.argsort(-probability, kind=\'stable\')\n    values, labels = probability[order], y[order]\n    ends = np.r_[np.flatnonzero(np.diff(values) != 0), len(values)-1]\n    tp = np.r_[0, np.cumsum(labels)[ends]]\n    fp = np.r_[0, (ends+1)-np.cumsum(labels)[ends]]\n    thresholds = np.r_[np.nextafter(float(values[0]), np.inf), values[ends]]\n    fn, tn = y.sum()-tp, (len(y)-y.sum())-fp\n    specificity = tn / (tn+fp)\n    f2 = np.divide(5*tp, 5*tp+4*fn+fp, out=np.zeros(len(tp), dtype=float), where=(5*tp+4*fn+fp)>0)\n    valid = np.flatnonzero(specificity >= minimum_specificity)\n    best = max(valid, key=lambda i: (f2[i], specificity[i], thresholds[i]))\n    return {\'threshold\': float(thresholds[best]), \'selection_role\': \'threshold\',\n            \'minimum_specificity\': minimum_specificity, \'selection_specificity\': float(specificity[best]),\n            \'selection_f2\': float(f2[best]),\n            \'note\': \'Development constraint, not a clinical recommendation or guarantee on evaluation data\'}\n\n\ndef calibration_diagnostics(y, probability):\n    y = np.asarray(y, dtype=float)\n    x = logit(np.clip(probability, 1e-6, 1-1e-6))\n    if len(np.unique(y)) < 2:\n        return {\'calibration_in_the_large\': None, \'calibration_intercept\': None, \'calibration_slope\': None}\n    def objective(theta, slope_fixed=False):\n        z = theta[0]+(x if slope_fixed else theta[1]*x)\n        residual = expit(z)-y\n        gradient = np.array([residual.mean()]) if slope_fixed else np.array([residual.mean(), (residual*x).mean()])\n        return float(np.mean(np.logaddexp(0, z)-y*z)), gradient\n    fixed = minimize(lambda b: objective(b, True), [0.], jac=True, method=\'L-BFGS-B\', bounds=[(-30,30)])\n    joint = minimize(objective, [0.,1.], jac=True, method=\'L-BFGS-B\', bounds=[(-30,30),(-30,30)])\n    return {\'calibration_in_the_large\': float(fixed.x[0]) if fixed.success else None,\n            \'calibration_intercept\': float(joint.x[0]) if joint.success else None,\n            \'calibration_slope\': float(joint.x[1]) if joint.success else None,\n            \'diagnostic_only\': \'Fitted for assessment; does not update the prediction model\',\n            \'bounded_fit\': \'Coefficients bounded to [-30,30]; inspect separation/constant predictions before interpretation\'}\n\n\ndef metrics(y, probability, threshold):\n    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)\n    pred = probability >= threshold\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()\n    both = len(np.unique(y)) == 2\n    return {\'images\': len(y), \'positive_images\': int(y.sum()), \'threshold\': float(threshold),\n            \'precision\': float(precision_score(y, pred, zero_division=0)),\n            \'recall\': float(recall_score(y, pred, zero_division=0)),\n            \'specificity\': float(tn/(tn+fp)) if tn+fp else None,\n            \'f1\': float(f1_score(y, pred, zero_division=0)),\n            \'f2\': float(fbeta_score(y, pred, beta=2, zero_division=0)),\n            \'roc_auc\': float(roc_auc_score(y, probability)) if both else None,\n            \'average_precision\': float(average_precision_score(y, probability)) if both else None,\n            \'brier\': float(brier_score_loss(y, probability)),\n            \'log_loss\': float(log_loss(y, probability, labels=[0,1])),\n            \'referral_fraction\': float(pred.mean()), \'tn\': int(tn), \'fp\': int(fp), \'fn\': int(fn), \'tp\': int(tp)}\n\n\ndef paired_patient_bootstrap(frame, scores, thresholds, *, repeats=200, seed=42):\n    """Resample whole patients with multiplicity, paired across every model."""\n    groups = [np.asarray(v) for v in frame.reset_index(drop=True).groupby(\'patient_id\', sort=True).indices.values()]\n    rng, samples = np.random.default_rng(seed), {name: [] for name in scores}\n    valid = 0\n    keys = [\'precision\', \'recall\', \'specificity\', \'f1\', \'f2\', \'roc_auc\', \'average_precision\', \'brier\', \'referral_fraction\']\n    for _ in range(repeats):\n        ix = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])\n        y = frame.target.to_numpy()[ix]\n        if len(np.unique(y)) != 2:\n            continue\n        valid += 1\n        for name, probability in scores.items():\n            samples[name].append(metrics(y, np.asarray(probability)[ix], thresholds[name][\'threshold\']))\n    if valid < max(2, int(.8*repeats)):\n        raise ValueError(\'Too few valid patient-bootstrap samples\')\n    intervals = {name: {key: np.quantile([r[key] for r in records], [.025,.975]).tolist() for key in keys}\n                 for name, records in samples.items()}\n    differences = {}\n    if \'Fusion\' in samples and \'DL\' in samples:\n        differences = {key: np.quantile([f[key]-d[key] for f,d in zip(samples[\'Fusion\'], samples[\'DL\'])],\n                                        [.025,.975]).tolist() for key in keys}\n    return {\'method\': \'Paired percentile patient-cluster bootstrap; image-weighted estimand\',\n            \'repeats_requested\': repeats, \'repeats_valid\': valid, \'seed\': seed,\n            \'ci95\': intervals, \'fusion_minus_dl_ci95\': differences,\n            \'limitation\': \'Conditional on fitted models; no training variability or guarantee against prior test reuse\'}\n', 'image_training.py': '"""EfficientNet training with persistent checkpoints and exact keyed predictions."""\nfrom __future__ import annotations\n\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom sklearn.metrics import average_precision_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\nfrom torchvision.models import EfficientNet_B0_Weights, efficientnet_b0\n\nfrom fusion_protocol import TARGET, write_json, sha256\n\n\ndef seed_worker(_):\n    seed = torch.initial_seed() % (2**32)\n    np.random.seed(seed)\n    random.seed(seed)\n\n\nclass ImageRecords(Dataset):\n    def __init__(self, frame, images, transform):\n        self.frame = frame.reset_index(drop=True)\n        self.images, self.transform = Path(images), transform\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, index):\n        row = self.frame.iloc[index]\n        with Image.open(self.images / row.cxr_filename) as source:\n            tensor = self.transform(source.convert(\'RGB\'))\n        return tensor, np.float32(row[TARGET]), str(row.patient_id), str(row.cxr_filename)\n\n\ndef transforms_for_run():\n    weights = EfficientNet_B0_Weights.DEFAULT.transforms()\n    normalise = transforms.Normalize(mean=weights.mean, std=weights.std)\n    # A conservative first protocol revision: same spatial crop in train/eval.\n    train = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),\n        transforms.RandomRotation(5), transforms.ToTensor(), normalise])\n    evaluation = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),\n        transforms.ToTensor(), normalise])\n    return train, evaluation\n\n\ndef loader(frame, images, *, batch_size=32, workers=2, training=False, generator=None):\n    train_transform, eval_transform = transforms_for_run()\n    return DataLoader(ImageRecords(frame, images, train_transform if training else eval_transform),\n        batch_size=batch_size, shuffle=training, num_workers=workers,\n        pin_memory=torch.cuda.is_available(), persistent_workers=False,\n        worker_init_fn=seed_worker, generator=generator)\n\n\n@torch.inference_mode()\ndef predict_images(model, frame, images, *, batch_size=32, workers=2):\n    model.eval()\n    device = next(model.parameters()).device\n    records = []\n    for tensors, targets, patients, filenames in loader(frame, images, batch_size=batch_size, workers=workers):\n        probabilities = torch.sigmoid(model(tensors.to(device, non_blocking=True))).flatten().cpu().numpy()\n        records.extend({\'patient_id\': str(p), \'cxr_filename\': str(f), \'target\': int(y),\n                        \'dl_probability\': float(score)} for p,f,y,score in zip(patients,filenames,targets,probabilities))\n    return pd.DataFrame(records)\n\n\ndef save_torch_atomic(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix+\'.tmp\')\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef train_image_model(frame, images, output, *, seed=42, epochs=10, patience=3,\n                      batch_size=32, workers=2, learning_rate=1e-4, resume=False):\n    output = Path(output)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if device.type == \'cuda\':\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    generator = torch.Generator().manual_seed(seed)\n    training, validation = frame[frame.role.eq(\'train\')], frame[frame.role.eq(\'branch_val\')]\n    config = {\'seed\': seed, \'epochs\': epochs, \'patience\': patience, \'batch_size\': batch_size,\n              \'workers\': workers, \'learning_rate\': learning_rate, \'architecture\': \'EfficientNet-B0\',\n              \'weights\': \'IMAGENET1K_V1\', \'manifest_sha256\': sha256(output/\'private/protocol_manifest.csv\'),\n              \'image_transform\': \'resize256 center224; train rotation5; ImageNet normalisation\',\n              \'feature_source_sha256\': sha256(Path(__file__).with_name(\'fusion_protocol.py\')),\n              \'image_source_sha256\': sha256(__file__)}\n    last_path, best_path = output/\'models/image_last.pt\', output/\'models/image_best.pt\'\n    state = None\n    if resume:\n        if not last_path.is_file():\n            raise FileNotFoundError(\'Resume requested but image_last.pt does not exist\')\n        # Load only checkpoints created by this code from your own private run.\n        state = torch.load(last_path, map_location=\'cpu\', weights_only=True)\n        if state[\'config\'] != config:\n            raise ValueError(\'Checkpoint config/source/split differs; start a separate run\')\n    elif last_path.exists():\n        raise FileExistsError(\'Existing image run; use resume=True or a new RUN_ID\')\n    model = efficientnet_b0(weights=None if state else EfficientNet_B0_Weights.DEFAULT)\n    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)\n    model.to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)\n    amp = torch.amp.GradScaler(\'cuda\', enabled=device.type == \'cuda\')\n    positives = int(training[TARGET].sum())\n    if positives == 0 or positives == len(training):\n        raise ValueError(\'Image training needs both classes\')\n    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(len(training)-positives)/positives], device=device))\n    history, best_ap, stale, start_epoch = [], -1., 0, 1\n    if state:\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        amp.load_state_dict(state[\'amp\'])\n        history, best_ap, stale, start_epoch = state[\'history\'], state[\'best_ap\'], state[\'stale\'], state[\'epoch\']+1\n        random.setstate(state[\'python_rng\'])\n        nr = state[\'numpy_rng\']\n        np.random.set_state((nr[0], np.array(nr[1], dtype=np.uint32), nr[2], nr[3], nr[4]))\n        torch.set_rng_state(state[\'torch_rng\'])\n        generator.set_state(state[\'loader_rng\'])\n        if device.type == \'cuda\' and state.get(\'cuda_rng\') is not None:\n            torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n    train_loader = loader(training, images, batch_size=batch_size, workers=workers,\n                          training=True, generator=generator)\n    write_json(output/\'image_training_config.json\', {**config, \'device\': str(device),\n               \'cuda\': torch.version.cuda, \'cudnn\': torch.backends.cudnn.version(),\n               \'gpu\': torch.cuda.get_device_name(0) if device.type == \'cuda\' else None,\n               \'positive_weight\': (len(training)-positives)/positives,\n               \'reproducibility\': \'Seeds and RNG states saved; cross-device/version bitwise identity is not guaranteed\'})\n    for epoch in range(start_epoch, epochs+1):\n        if stale >= patience:\n            break\n        model.train()\n        accumulated = 0.\n        for tensors, labels, _, _ in train_loader:\n            tensors = tensors.to(device, non_blocking=True)\n            labels = labels.to(device, non_blocking=True).view(-1,1)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=device.type == \'cuda\'):\n                loss = criterion(model(tensors), labels)\n            if not torch.isfinite(loss):\n                raise RuntimeError(\'Nonfinite image loss; last completed epoch remains saved\')\n            amp.scale(loss).backward()\n            amp.step(optimizer)\n            amp.update()\n            accumulated += float(loss.detach())*len(tensors)\n        predictions = predict_images(model, validation, images, batch_size=batch_size, workers=workers)\n        ap = float(average_precision_score(predictions.target, predictions.dl_probability))\n        history.append({\'epoch\': epoch, \'train_loss\': accumulated/len(training), \'branch_val_ap\': ap})\n        if ap > best_ap:\n            best_ap, stale = ap, 0\n            save_torch_atomic(best_path, {\'model\': model.state_dict(), \'epoch\': epoch, \'branch_val_ap\': ap, \'config\': config})\n        else:\n            stale += 1\n        nr = np.random.get_state()\n        save_torch_atomic(last_path, {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n            \'amp\': amp.state_dict(), \'epoch\': epoch, \'history\': history, \'best_ap\': best_ap,\n            \'stale\': stale, \'config\': config, \'python_rng\': random.getstate(),\n            \'numpy_rng\': (nr[0], nr[1].tolist(), nr[2], nr[3], nr[4]),\n            \'torch_rng\': torch.get_rng_state(), \'loader_rng\': generator.get_state(),\n            \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else None})\n        pd.DataFrame(history).to_csv(output/\'public/dl_training_history.csv\', index=False)\n        print(f\'Epoch {epoch}: loss={history[-1]["train_loss"]:.4f}, branch-val AP={ap:.4f}; checkpoint saved\')\n    best = torch.load(best_path, map_location=device, weights_only=True)\n    model.load_state_dict(best[\'model\'])\n    write_json(output/\'public/image_checkpoint_receipt.json\',\n               {\'best_epoch\': best[\'epoch\'], \'branch_val_ap\': best[\'branch_val_ap\'],\n                \'checkpoint_sha256\': sha256(best_path), \'role_used_for_selection\': \'branch_val\'})\n    return model\n', 'run_support.py': '"""Drive-friendly persistence and explicit evaluation helpers."""\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.calibration import calibration_curve\n\nfrom fusion_protocol import (FEATURES, KEYS, TARGET, calibrated_scores, calibration_diagnostics,\n    environment_receipt, fit_calibrators, fit_fusion, metrics, pair_predictions,\n    paired_patient_bootstrap, raw_scores, select_threshold, sha256, write_json)\n\n\ndef dataset_root(project, cache):\n    project, cache = Path(project), Path(cache)\n    def discover(parent):\n        if not parent.exists():\n            return None\n        for metadata in parent.rglob(\'metadata.csv\'):\n            if (metadata.parent/\'images\').is_dir():\n                return metadata.parent\n        return None\n    existing = discover(cache) or discover(project)\n    if existing:\n        return existing\n    candidates = sorted(project.glob(\'*.zip\'))\n    for path in candidates:\n        with zipfile.ZipFile(path) as archive:\n            if not any(Path(name).name == \'metadata.csv\' for name in archive.namelist()):\n                continue\n            cache.mkdir(parents=True, exist_ok=True)\n            for info in archive.infolist():\n                dest = (cache/info.filename).resolve()\n                if not dest.is_relative_to(cache.resolve()) or \'\\\\\' in info.filename:\n                    raise ValueError(\'Unsafe archive path\')\n                if info.external_attr >> 16 & 0o170000 == 0o120000:\n                    raise ValueError(\'Symlinks are not allowed in dataset archive\')\n            archive.extractall(cache)\n        found = discover(cache)\n        if found:\n            return found\n    raise FileNotFoundError(\'No dataset with metadata.csv + images/ found; place the official ZIP in the project folder\')\n\n\ndef initialise_run(output, config, *, resume=False):\n    output = Path(output)\n    config_path = output/\'run_config.json\'\n    if config_path.exists():\n        previous = json.loads(config_path.read_text())\n        if not resume or previous != config:\n            raise ValueError(\'Run exists or config changed. Use matching RESUME=True or a new RUN_ID\')\n    else:\n        if resume:\n            raise FileNotFoundError(\'Cannot resume a run that has no receipt\')\n        output.mkdir(parents=True, exist_ok=False)\n        write_json(config_path, config)\n    for name in [\'private\', \'models\', \'public\']:\n        (output/name).mkdir(exist_ok=True)\n    environment = environment_receipt()\n    prior = output/\'environment.json\'\n    if resume and prior.exists() and json.loads(prior.read_text()) != environment:\n        raise ValueError(\'Runtime environment changed; start a new run for reproducible resume\')\n    write_json(prior, environment)\n    pinned = [f\'{name}=={version}\' for name,version in environment[\'packages\'].items() if version]\n    (output/\'requirements-runtime.txt\').write_text(\'\\n\'.join(pinned)+\'\\n\')\n    return output\n\n\ndef artifact_receipt(output):\n    output = Path(output)\n    files = {str(path.relative_to(output)).replace(\'\\\\\',\'/\'): sha256(path)\n             for path in output.rglob(\'*\') if path.is_file() and path.name != \'artifact_receipt.json\' and not path.name.endswith(\'.tmp\')}\n    write_json(output/\'artifact_receipt.json\', {\'recorded_utc\': datetime.now(timezone.utc).isoformat(),\n        \'artifacts_sha256\': files, \'private_files\': \'Contains protected record keys, predictions and model artifacts. Do not upload entire run to public GitHub.\'})\n\n\ndef fit_late_stage(development_predictions, output, *, seed=42, minimum_specificity=.80):\n    frame = development_predictions\n    if not set(frame.role).issubset({\'fusion_fit\',\'calibration\',\'threshold\',\'branch_val\'}):\n        raise ValueError(\'Evaluation rows cannot be used to fit late stages\')\n    if frame.groupby(\'patient_id\').role.nunique().max() != 1:\n        raise ValueError(\'Patient overlap in late-stage inputs\')\n    fusion = fit_fusion(frame, seed=seed)\n    calibrators = fit_calibrators(frame, fusion, seed=seed)\n    tuning = frame[frame.role.eq(\'threshold\')]\n    probabilities = calibrated_scores(tuning, fusion, calibrators)\n    thresholds = {name: select_threshold(tuning.target, scores, minimum_specificity=minimum_specificity)\n                  for name,scores in probabilities.items()}\n    bundle = {\'fusion\': fusion, \'calibrators\': calibrators, \'thresholds\': thresholds}\n    joblib.dump(bundle, Path(output)/\'models/fusion_bundle.joblib\')\n    write_json(Path(output)/\'public/selected_thresholds.json\', thresholds)\n    write_json(Path(output)/\'public/stage_roles.json\', {\n        \'branch_training\': \'train\', \'branch_selection\': \'branch_val\', \'fusion_fitting\': \'fusion_fit\',\n        \'sigmoid_calibration\': \'calibration\', \'threshold_choice\': \'threshold\',\n        \'evaluation\': \'No evaluation data used in any fit; historical test is not an untouched external test\',\n        \'natural_prevalence\': \'No resampling or balanced weights for fusion/calibration\',\n        \'target_specificity\': minimum_specificity})\n    artifact_receipt(output)\n    return bundle\n\n\ndef evaluate_predictions(frame, bundle, output, *, label, repeats=200, seed=42):\n    if label not in {\'historical_comparison\', \'external_evaluation\'}:\n        raise ValueError(\'Explicit evaluation label required\')\n    output = Path(output)\n    manifest = pd.read_csv(output/\'private/protocol_manifest.csv\',dtype={\'patient_id\':str,\'cxr_filename\':str})\n    if frame[KEYS].isna().any().any() or frame.duplicated(KEYS).any() or not frame.target.isin([0,1]).all():\n        raise ValueError(\'Invalid evaluation keys or targets\')\n    if label == \'historical_comparison\':\n        if \'role\' not in frame or not frame.role.eq(\'historical_test\').all():\n            raise ValueError(\'Historical evaluation accepts historical_test rows only\')\n        expected = manifest[manifest.role.eq(\'historical_test\')][[*KEYS,TARGET]].rename(columns={TARGET:\'expected_target\'})\n        aligned = frame.merge(expected,on=KEYS,how=\'outer\',indicator=True,validate=\'one_to_one\')\n        if not aligned[\'_merge\'].eq(\'both\').all() or not aligned.target.eq(aligned.expected_target).all():\n            raise ValueError(\'Historical evaluation must match the entire recorded historical cohort and labels\')\n    elif frame.cxr_filename.isin(manifest.cxr_filename).any():\n        raise ValueError(\'External image filenames overlap with the development dataset\')\n    report_path = output/f\'public/{label}_metrics.csv\'\n    if report_path.exists():\n        raise FileExistsError(\'Evaluation already written for this run; do not silently overwrite it\')\n    scores = calibrated_scores(frame, bundle[\'fusion\'], bundle[\'calibrators\'])\n    raw = raw_scores(frame, bundle[\'fusion\'])\n    rows = []\n    prediction_export = frame[[*KEYS,\'target\']].copy()\n    curves = []\n    for name, probability in scores.items():\n        rows.append({\'model\': name, \'evaluation\': label,\n                     **metrics(frame.target, probability, bundle[\'thresholds\'][name][\'threshold\']),\n                     **calibration_diagnostics(frame.target, probability)})\n        prediction_export[f\'{name.lower()}_raw\'] = raw[name]\n        prediction_export[f\'{name.lower()}_calibrated\'] = probability\n        observed, predicted = calibration_curve(frame.target, probability, n_bins=10, strategy=\'quantile\')\n        curves.extend({\'model\': name, \'mean_predicted\': float(p), \'observed_fraction\': float(o)}\n                      for p,o in zip(predicted,observed))\n    # Complete statistical calculations before committing the evaluation report.\n    ci = paired_patient_bootstrap(frame, scores, bundle[\'thresholds\'], repeats=repeats, seed=seed)\n    pd.DataFrame(rows).to_csv(report_path, index=False)\n    raw_rows = [{\'model\':name,\'evaluation\':label,\n                 **metrics(frame.target,probability,.5),**calibration_diagnostics(frame.target,probability)}\n                for name,probability in raw.items()]\n    pd.DataFrame(raw_rows).to_csv(output/f\'public/{label}_uncalibrated_at_05.csv\',index=False)\n    pd.DataFrame(curves).to_csv(output/f\'public/{label}_calibration_curve.csv\', index=False)\n    prediction_export.to_csv(output/f\'private/{label}_predictions.csv\', index=False)\n    write_json(output/f\'public/{label}_ci95.json\', ci)\n    write_json(output/f\'public/{label}_receipt.json\', {\'label\': label, \'images\': len(frame),\n        \'patients\': frame.patient_id.nunique(), \'positive_images\': int(frame.target.sum()),\n        \'predictions_sha256\': sha256(output/f\'private/{label}_predictions.csv\'),\n        \'model_bundle_sha256\': sha256(output/\'models/fusion_bundle.joblib\'),\n        \'manifest_sha256\': sha256(output/\'private/protocol_manifest.csv\'),\n        \'historical_test_previously_inspected\': label == \'historical_comparison\',\n        \'claim\': \'Historical comparison only; cannot establish new unbiased test performance\' if label == \'historical_comparison\'\n                 else \'Independent source must be justified by the supplied external-source receipt; software checks cannot prove independence\'})\n    artifact_receipt(output)\n    return pd.DataFrame(rows)\n'}
for filename,source in MODULE_SOURCES.items():
    (MODULE_ROOT/filename).write_text(source, encoding='utf-8')
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0,str(MODULE_ROOT))
from fusion_protocol import *
from run_support import *
from image_training import train_image_model, predict_images
print('Embedded source modules ready; no helper upload needed.')


## 1 Configure a new run
Leave `RUN_ID=None` for a fresh run. To resume an interrupted training run, paste its exact folder name and set `RESUME=True`.
Resume checks the source, environment, split and training configuration. Do not switch metadata variant midway.
The default specificity target 0.80 is an experimental development constraint, not a clinical recommendation.

In [ ]:
RUN_ID = None  # e.g. '20261117T090000Z_ab12cd34'
RESUME = False
SEED = 42
FEATURE_SET = 'age_sex'  # or 'full_retrospective' for a separately documented ablation
MINIMUM_SPECIFICITY = 0.80
EPOCHS, PATIENCE, BATCH_SIZE, NUM_WORKERS = 10, 3, 32, 2
LEARNING_RATE = 1e-4
BOOTSTRAP_REPEATS = 200  # increase for a final analysis; keep plan fixed before evaluation
if RESUME and RUN_ID is None:
    raise ValueError('Set the exact previous RUN_ID before resuming')
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
if Path(RUN_ID).name != RUN_ID or '/' in RUN_ID or '\\' in RUN_ID:
    raise ValueError('RUN_ID must be a plain folder name')
DATA_ROOT = dataset_root(PROJECT,CACHE)
IMAGES = DATA_ROOT/'images'
historical_candidates = [PROJECT/'outputs_ml_support_dl/metadata_with_patient_split.csv',
                          PROJECT/'processed/metadata_with_split.csv']
SOURCE_MANIFEST = next((p for p in historical_candidates if p.is_file()), DATA_ROOT/'metadata.csv')
raw = pd.read_csv(SOURCE_MANIFEST, dtype={'patient_id':str,'cxr_filename':str})
manifest = make_protocol(raw,seed=SEED)
missing_images = sum(not (IMAGES/name).is_file() for name in manifest.cxr_filename)
if missing_images:
    raise FileNotFoundError(f'{missing_images} referenced images are missing')
config = {'protocol':'v2_patient_separated', 'seed':SEED, 'feature_set':FEATURE_SET,
          'source_metadata_sha256':sha256(DATA_ROOT/'metadata.csv'),
          'source_manifest_sha256':sha256(SOURCE_MANIFEST),
          'historical_split_source':'saved manifest' if SOURCE_MANIFEST.name != 'metadata.csv' else 'reconstructed old 70/15/15 seed42 split',
          'minimum_specificity':MINIMUM_SPECIFICITY,'epochs':EPOCHS,'patience':PATIENCE,
          'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,'learning_rate':LEARNING_RATE,
          'source_sha256':{name:sha256(MODULE_ROOT/name) for name in MODULE_SOURCES}}
RUN = initialise_run(PROJECT/'outputs_ml_support_dl_v2'/RUN_ID,config,resume=RESUME)
serial = manifest.to_csv(index=False,lineterminator='\n').encode('utf-8')
manifest_file = RUN/'private/protocol_manifest.csv'
if RESUME and manifest_file.read_bytes() != serial:
    raise ValueError('Protocol manifest differs from the saved run')
if not RESUME:
    manifest_file.write_bytes(serial)
write_json(RUN/'public/split_summary.json',split_summary(manifest))
write_json(RUN/'public/evidence_policy.json',{'previous_test_seen':True,
    'default_metadata':FEATURE_SET,'independent_external_evaluation_available':False,
    'raw_image_archive_checksum':'not computed by this cell; metadata and split hashes are recorded',
    'dataset_access':'Use only your authorised PhysioNet dataset; keep private artifacts off public repositories'})
print('RUN_ID:',RUN_ID)
print('Persistent run:',RUN)
display(pd.DataFrame(split_summary(manifest)))
artifact_receipt(RUN)


## 2 Train metadata models on train and select on branch_val
The original train and historical test assignments are preserved. Original validation patients are assigned once to
`branch_val` (25%), `fusion_fit` (35%), `calibration` (20%) and `threshold` (20%). All image encounters from a patient follow that role.
Neither fusion-fit, calibration nor threshold patients influence branch model selection or image early stopping.

In [ ]:
ml_artifact = RUN/'models/metadata_bundle.joblib'
if RESUME and ml_artifact.exists():
    ml = joblib.load(ml_artifact)  # own trusted private run only
else:
    ml = fit_metadata(manifest,seed=SEED,feature_set=FEATURE_SET)
    joblib.dump(ml,ml_artifact)
pd.DataFrame(ml['comparisons']).to_csv(RUN/'public/metadata_branch_selection.csv',index=False)
write_json(RUN/'public/metadata_model.json',{'selected_model':ml['name'],'features':ml['features'],
    'selection_role':'branch_val','feature_set':FEATURE_SET})
display(pd.DataFrame(ml['comparisons']))
artifact_receipt(RUN)


## 3 Train images and save after every completed epoch
Best and last checkpoints, optimizer/AMP/RNG state and history are written to Drive each epoch.
Early stopping uses only `branch_val`. Resume restarts at the last completed epoch.
It does not reuse old weights trained with the whole previous validation partition.

In [ ]:
image_model = train_image_model(manifest,IMAGES,RUN,seed=SEED,epochs=EPOCHS,patience=PATIENCE,
    batch_size=BATCH_SIZE,workers=NUM_WORKERS,learning_rate=LEARNING_RATE,resume=RESUME)
artifact_receipt(RUN)


## 4 Fit fusion then calibrate then choose thresholds on disjoint patients

In [ ]:
def predictions_for(frame):
    dl = predict_images(image_model,frame,IMAGES,batch_size=BATCH_SIZE,workers=NUM_WORKERS)
    ml_predictions = frame[[*KEYS,TARGET]].rename(columns={TARGET:'target'}).copy()
    ml_predictions['ml_probability'] = ml['model'].predict_proba(frame[ml['features']])[:,1]
    combined = pair_predictions(dl,ml_predictions)
    combined = combined.merge(frame[[*KEYS,'role']],on=KEYS,how='left',validate='one_to_one')
    return combined
development = manifest[manifest.role.isin(['fusion_fit','calibration','threshold'])].copy()
development_predictions = predictions_for(development)
development_predictions.to_csv(RUN/'private/development_predictions.csv',index=False)
bundle = fit_late_stage(development_predictions,RUN,seed=SEED,minimum_specificity=MINIMUM_SPECIFICITY)
print('Fitted fusion, calibrators and thresholds; no test predictions computed.')
display(pd.DataFrame(bundle['thresholds']).T)


## 5 Optional historical comparison
Disabled by default. The old test has already been viewed and cannot become a fresh test by changing the random seed.
If enabled, this reports historical-comparison metrics, calibration and paired patient-bootstrap CIs.
Do not tune using these values or treat them as independent confirmation of improvement.

In [ ]:
RUN_HISTORICAL_COMPARISON = False
if RUN_HISTORICAL_COMPARISON:
    historical = manifest[manifest.role.eq('historical_test')].copy()
    historical_predictions = predictions_for(historical)
    display(evaluate_predictions(historical_predictions,bundle,RUN,label='historical_comparison',
                                repeats=BOOTSTRAP_REPEATS,seed=SEED))
else:
    print('Historical evaluation skipped. Models and development evidence remain saved.')


## 6 Optional independent external evaluation
Requires a separately acquired and appropriately licensed cohort with the **same echo-derived outcome** and predictor definitions.
Another CXR disease label dataset is not interchangeable. Specify a stable source ID and document why patients are independent.
Different de-identified ID namespaces cannot prove independence; the operator's source justification is retained.
No automatic train/test resplitting is used to invent a fresh evaluation.

In [ ]:
EXTERNAL_METADATA = None  # private CSV with original required columns and paired images
EXTERNAL_IMAGES = None
EXTERNAL_SOURCE_ID = None
EXTERNAL_INDEPENDENCE_JUSTIFICATION = None
if EXTERNAL_METADATA is not None:
    if not all([EXTERNAL_IMAGES,EXTERNAL_SOURCE_ID,EXTERNAL_INDEPENDENCE_JUSTIFICATION]):
        raise ValueError('Document the external source, images and independence before evaluation')
    if str(EXTERNAL_SOURCE_ID).lower() in {'chexchonet','chexchonet-v1','chexchonet-1.0.0'}:
        raise ValueError('The old CheXchoNet cohort is not an independent external source')
    ext = prepare_frame(pd.read_csv(EXTERNAL_METADATA,dtype={'patient_id':str,'cxr_filename':str}))
    shared_keys = set(map(tuple,manifest[KEYS].to_numpy())) & set(map(tuple,ext[KEYS].to_numpy()))
    if shared_keys or ext.cxr_filename.isin(manifest.cxr_filename).any():
        raise ValueError('Possible repeated images in external source; resolve before evaluation')
    if any(not (Path(EXTERNAL_IMAGES)/name).is_file() for name in ext.cxr_filename):
        raise FileNotFoundError('Missing external images')
    ext['role'] = 'external_evaluation'
    dl = predict_images(image_model,ext,EXTERNAL_IMAGES,batch_size=BATCH_SIZE,workers=NUM_WORKERS)
    ext_ml = ext[[*KEYS,TARGET]].rename(columns={TARGET:'target'}).copy()
    ext_ml['ml_probability'] = ml['model'].predict_proba(ext[ml['features']])[:,1]
    external_predictions = pair_predictions(dl,ext_ml)
    write_json(RUN/'external_source.json',{'source_id':EXTERNAL_SOURCE_ID,
        'metadata_sha256':sha256(EXTERNAL_METADATA),'independence_justification':EXTERNAL_INDEPENDENCE_JUSTIFICATION,
        'identity_namespace_limitation':'No key collision does not establish patient independence across sites'})
    display(evaluate_predictions(external_predictions,bundle,RUN,label='external_evaluation',
                                repeats=BOOTSTRAP_REPEATS,seed=SEED))
else:
    print('No independent external cohort supplied; no external-performance claim made.')


## 7 Locate saved evidence
The persistent run is under `MyDrive/Hoc_may_2_0/DL_Xray/CheXchoNet/outputs_ml_support_dl_v2/<RUN_ID>/`.
- `models/`: ML, fusion/calibrators and best/last image checkpoints.
- `private/`: manifest and per-examination predictions; never publish these patient-level records.
- `public/`: aggregate summaries and any explicitly run evaluation outputs. Review them before sharing.
- Runtime, source/split/checkpoint hashes and model-selection criteria are saved in JSON receipts.
Keep `RUN_ID` to resume. Do not overwrite the historical run or its submitted report.